This is a notebook to learn autograd. It is not related to the project.

In [1]:
# install autograd
%pip install autograd

  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import autograd
import numpy as np

In [3]:
# define the function
def f(x):
  return (3.*x + 1)**2

In [4]:
df = autograd.grad(f)
print(f(2.), df(2.))


49.0 42.0


## Four things to notice:

- autograd.grad(f) returns a new function, not a number. You then call it at a point.
- You must import NumPy from autograd.numpy. That version is the one that records the operations.
- Plain NumPy runs silently, and the recording is lost.
- Inputs must be floats (2., not 2).
- grad expects a function that returns a single number.

In [5]:
# new function
def g(x):
  return (x**3 - 2.*x)

dg = autograd.grad(g)
print("g(x) at 2: ",g(2.))
print("dg(x) at 2: ", dg(2.))

g(x) at 2:  4.0
dg(x) at 2:  10.0


In [6]:
print(np.__name__)
print(np.exp)

numpy
<ufunc 'exp'>


In [7]:
def h(x):
    return np.exp(2.*x)

dh = autograd.grad(h)
print(dh(1.))

14.7781121978613


In [8]:
import importlib.metadata
print(importlib.metadata.version("autograd"))

1.9.1


In [9]:
# Two variables
import autograd.numpy as np
def d(v):
  return v[0]**2*v[1]

dv = autograd.grad(d)
print(dv(np.array([2., 3.])))


[12.  4.]


In [10]:
# Another way for the two variables
def p(a,b):
  return a*b**2

print(autograd.grad(p, 0)(2., 3.))
print(autograd.grad(p, 1)(2., 3.))

9.0
12.0


In [11]:
# Writing function for two variables
# Equation used in the rate equation
def r1(c):
  return 0.08*c[0]**0.7*c[1]**1.3

dr1 = autograd.grad(r1)
print(dr1(np.array([0.5,0.5])))


[0.028 0.052]


In [12]:
print(dr1(np.array([1.0,1.0])))

[0.056 0.104]


In [13]:
dr1(np.array([1.0,1.0]))/dr1(np.array([0.5,0.5]))

array([2., 2.])

In [14]:
# Another way
def r2(Ca,Cb):
  return 0.08*Ca**0.7*Cb**1.3

grad_Ca = autograd.grad(r2, 0)(0.5, 0.5)
grad_Cb = autograd.grad(r2, 1)(0.5, 0.5)

print(grad_Ca)
print(grad_Cb)

0.027999999999999997
0.052


In [15]:
# Jacobian. Function which returns a vector
def f(v):
  return np.array([v[0]*v[1], v[0] + v[1]**2])

print(autograd.jacobian(f)(np.array([2.,3.])))

[[3. 2.]
 [1. 6.]]


In [16]:
# Write a function similar to the reference
# parameters
tau = 100
C_Ain, C_Bin , C_Xin = 0.7, 0.3, 0
# using the rate law, the right hand side
def dcdt(c):
  c_A, c_B, c_X = c
  r = 0.08 * c_A**0.7 * c_B**1.3
  return np.array([(C_Ain - c_A)/tau - r,
                   (C_Bin - c_B)/tau - r,
                   (C_Xin - c_X)/tau + r])
print(autograd.jacobian(dcdt)(np.array([0.5,0.5,0])))

[[-0.038 -0.052  0.   ]
 [-0.028 -0.062  0.   ]
 [ 0.028  0.052 -0.01 ]]


In [ ]:
# Traing for a neural network for one input

def predict(params, x):
    w,b = params
    return 1/(1 + np.exp(-(w*x + b)))

def loss(params, x , target):
    return (predict(params, x) - target)**2

grad_loss = autograd.grad(loss)

print(grad_loss(np.array([0.5, 0.1]), 1.0, 0.8))

[-0.07062281 -0.07062281]


In [20]:
params = np.array([0.5, 0.1])
history = {}                          # renamed so it doesn't overwrite the loss function
grad_loss = autograd.grad(loss)       # created once, outside the loop

for i in range(100):
    params = params - 0.5 * grad_loss(params, 1.0, 0.8)
    if (i + 1) % 20 == 0:             # % gives the remainder
        history[f"loss_{i+1}"] = loss(params, 1.0, 0.8)   # the loss value, not the gradient function

for k, v in history.items():
    print(k, v)

print("final params:", params)
print("final prediction:", predict(params, 1.0))        
               

loss_20 0.0009190227627484372
loss_40 8.416281092190506e-05
loss_60 9.385570110304366e-06
loss_80 1.1118105118897929e-06
loss_100 1.3439393969566916e-07
final params: [0.89200235 0.49200235]
final prediction: 0.7996334022099144


In [22]:
# training for two input

def predict(params, x1, x2):
    w1, w2,b = params
    return 1/(1 + np.exp(-(w1*x1 + w2*x2+ b)))

def loss(params, x1,x2 , target):
    return (predict(params, x1, x2) - target)**2

grad_loss = autograd.grad(loss)
print(grad_loss(np.array([0.5,0.5, 0.1]), 1.0, 2.0, 0.8))

[0.00895002 0.01790004 0.00895002]


In [26]:
# rerun for the gradient descent
def predict(params, x):
    w = params[:2]     # weight vector
    b = params[2]      # bias
    z = np.dot(w,x) + b            # the weighted sum, using np.dot(w, x) and b
    return 1/(1 + np.exp(-(z)))

def loss(params, x , target):
    return (predict(params, x) - target)**2

params = np.array([0.5, 0.5, 0.1])
history = {}                          # renamed so it doesn't overwrite the loss function
grad_loss = autograd.grad(loss)       # created once, outside the loop
x = np.array([1.0, 2.0])
for i in range(100):
    params = params - 0.5 * grad_loss(params, x, 0.8)
    if (i + 1) % 20 == 0:             # % gives the remainder
        history[f"loss_{i+1}:"] = loss(params,x, 0.8)   # the loss value, not the gradient function

for k, v in history.items():
    print(k, v)

print("final params:", params)
print("final prediction:", predict(params, x))        



loss_20: 2.3501552810992886e-06
loss_40: 3.055783984822459e-09
loss_60: 3.877425575893822e-12
loss_80: 4.915681239334489e-15
loss_100: 6.2317550858259925e-18
final params: [0.4643824  0.42876479 0.0643824 ]
final prediction: 0.8000000024963484
